# 🚀 DataMate: Dedicated Runner for Dual NVIDIA RTX 5090 (Vast.ai)
This notebook is custom-configured for **Dual NVIDIA RTX 5090 (64GB VRAM total)** instances on **Vast.ai** (US / Taiwan / EU Datacenters).
- **Hardware:** 2x RTX 5090 (Blackwell 32GB GDDR7 each, 1.8 TB/s memory bandwidth)
- **Engine:** vLLM with Tensor Parallelism (`VLLM_TP=2` across both GPUs)
- **Direct High-Speed Network:** Pulls directly from official GitHub and Hugging Face repositories at full 1Gbps - 10Gbps line speed!
- **Estimated Benchmark Runtime:** ~10-15 minutes for 500 questions!


## 1. Hardware & Disk Space Guard (Pre-Flight Checks)
Verifies Dual RTX 5090 detection, CUDA 12.6+/13.x, and ensures disk space is $\ge$ 25GB.


In [ ]:
import os
import shutil
import torch

print("=" * 60)
print("🔍 HARDWARE & PRE-FLIGHT VERIFICATION")
print("=" * 60)
!nvidia-smi

# Check GPU Count and VRAM
if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    print(f"\n✅ Detected {gpu_count} NVIDIA GPU(s):")
    for i in range(gpu_count):
        name = torch.cuda.get_device_name(i)
        vram = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"   [GPU {i}] {name} - {vram:.1f} GB VRAM")
else:
    print("❌ No CUDA GPU detected!")

# Check Disk Space
total, used, free = shutil.disk_usage(".")
free_gb = free / (1024**3)
total_gb = total / (1024**3)
print(f"\n💾 Disk Space: {free_gb:.1f} GB free out of {total_gb:.1f} GB total")
if free_gb < 25:
    print("⚠️ WARNING: Free disk space is less than 25 GB! Model download may fail if disk runs out.")
    print("👉 If you rented 16GB disk, please increase Disk Space to 60-80 GB in Vast.ai configuration.")
else:
    print("✅ Disk space is sufficient for model and pipeline.")
print("=" * 60)


## 2. Clone / Update DataMate Repository
Fetches the latest Phase 10 / Phase 11 code on branch `loc` using fast shallow clone (`--depth 1`).


In [ ]:
import os
REPO_URL = "https://github.com/vohoangloc-32/datamate.git"
BRANCH = "loc"

# Ensure working directory is datamate repository
if os.path.exists("src/predict.py"):
    print("✅ Already inside DataMate repository.")
elif os.path.exists("datamate/src/predict.py"):
    print("📂 Found datamate directory. Changing working directory to datamate...")
    %cd datamate
else:
    print("⬇️ Cloning DataMate repository into 'datamate' directory...")
    !rm -rf datamate
    !git clone --depth 1 -b {BRANCH} {REPO_URL} datamate
    %cd datamate

print("🔄 Pulling latest updates from branch loc...")
!git fetch origin {BRANCH} --depth 1
!git checkout {BRANCH}
!git reset --hard origin/{BRANCH}

print("\n📌 Current Commit:")
!git log -1 --oneline


## 3. Install Dependencies & Verify Pipeline
Installs vLLM, acceleration dependencies, and executes the full 196 unit test suite.


In [ ]:
print("📦 Cleaning conflicting packages & installing requirements...")
!pip uninstall -y torchaudio
!pip install -q vllm openpyxl xlrd tabulate pytest huggingface_hub pandas hf-transfer

print("\n🧪 Running full test suite...")
!pytest tests/ -q


## 4. Download Model Weights Directly from Hugging Face
Downloads `Qwen/Qwen2.5-Coder-14B-Instruct-GPTQ-Int8` directly with parallel acceleration (`hf_transfer`).


In [ ]:
import os
import time
from huggingface_hub import snapshot_download

MODEL_REPO = "Qwen/Qwen2.5-Coder-14B-Instruct-GPTQ-Int8"
LOCAL_MODEL_DIR = os.path.expanduser("~/model_cache/Qwen2.5-Coder-14B-Instruct-GPTQ-Int8")
os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)

# Enable direct high-speed transfer
os.environ.pop("HF_ENDPOINT", None)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print(f"⬇️ Downloading {MODEL_REPO} to {LOCAL_MODEL_DIR}...")
t0 = time.time()
snapshot_download(
    repo_id=MODEL_REPO,
    local_dir=LOCAL_MODEL_DIR,
    local_dir_use_symlinks=False,
    max_workers=8
)
t_dl = time.time() - t0
print(f"✅ Model weights ready in {t_dl:.1f}s ({t_dl/60:.2f} min)!")


## 5. Run DataMate Prediction Pipeline (Dual 5090 Optimized)
Runs inference using vLLM Tensor Parallelism (`VLLM_TP=2`) across both RTX 5090 GPUs.
- Context window: 16,384 tokens
- Expected duration: **~10 - 15 minutes** for 500 questions!


In [ ]:
#@title ⚙️ Pipeline Execution: Dual RTX 5090 Optimized { run: "auto" }
import os
import time
import torch

# Multi-GPU / Tensor Parallelism for Dual RTX 5090 (64GB VRAM)
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
os.environ["VLLM_TP"] = "2" if gpu_count >= 2 else "1"
os.environ["VLLM_DTYPE"] = "auto"
os.environ["VLLM_MAX_MODEL_LEN"] = "16384"
os.environ["VLLM_MAX_NEW_TOKENS"] = "1024"

# Execution mode: 'train_unseen_500' | 'train_500' | 'public_test' | 'train_200'
MODE = "train_unseen_500" #@param ["train_unseen_500", "train_500", "public_test", "train_200"]
CONFIDENCE_THRESHOLD = "0.45" #@param ["0.45", "0.40", "0.50", "0.35"]

WORKDIR = os.path.abspath(".")
if MODE == "public_test":
    DATA_DIR = os.path.join(WORKDIR, "data/public_test")
    QUESTIONS_FILE = os.path.join(WORKDIR, "data/public_test/questions.jsonl")
elif MODE == "train_unseen_500":
    DATA_DIR = os.path.join(WORKDIR, "data/train")
    QUESTIONS_FILE = os.path.join(WORKDIR, "data/sample_questions_unseen_500.jsonl")
elif MODE == "train_500":
    DATA_DIR = os.path.join(WORKDIR, "data/train")
    QUESTIONS_FILE = os.path.join(WORKDIR, "data/sample_questions_500.jsonl")
elif MODE == "train_200":
    DATA_DIR = os.path.join(WORKDIR, "data/train")
    QUESTIONS_FILE = os.path.join(WORKDIR, "data/sample_questions_200.jsonl")
else:
    raise ValueError(f"Unknown MODE: {MODE}")

OUTPUT_DIR = os.path.join(WORKDIR, "results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 60)
print("🚀 DATAMATE DUAL RTX 5090 INFERENCE ENGINE")
print("=" * 60)
print(f"GPUs Detected     : {gpu_count} (Tensor Parallel = {os.environ['VLLM_TP']})")
print(f"Execution Mode    : {MODE}")
print(f"Data Directory    : {DATA_DIR}")
print(f"Questions File    : {QUESTIONS_FILE}")
print(f"Output Directory  : {OUTPUT_DIR}")
print(f"Model Directory   : {LOCAL_MODEL_DIR}")
print(f"Confidence Thresh : {CONFIDENCE_THRESHOLD}")
print("=" * 60)

t_start = time.time()
!python src/predict.py \
    --data-dir {DATA_DIR} \
    --questions-file {QUESTIONS_FILE} \
    --output-dir {OUTPUT_DIR} \
    --model-dir {LOCAL_MODEL_DIR} \
    --confidence-threshold {CONFIDENCE_THRESHOLD} \
    --log-level INFO

elapsed = time.time() - t_start
print(f"\n🎉 Pipeline finished in {elapsed:.1f} seconds ({elapsed/60:.2f} minutes)!")


## 6. Compute Official Penalized Accuracy (PA) & Benchmark Results
Evaluates predictions against ground truth answers using the competition metric:
$$\text{PA} = \frac{\text{Correct} - \text{Wrong}}{\text{Total}}$$


In [ ]:
import os
import json
import pandas as pd
from src.evaluate import compute_pa

SUBMISSION_PATH = os.path.join(OUTPUT_DIR, "submission.jsonl")
ANSWERS_PATH = os.path.join(DATA_DIR, "answers.jsonl")

if os.path.exists(SUBMISSION_PATH):
    submission_records = [json.loads(l) for l in open(SUBMISSION_PATH, encoding="utf-8-sig") if l.strip()]
    print(f"✅ Generated submission file: {SUBMISSION_PATH} with {len(submission_records)} records.")
    
    if os.path.exists(ANSWERS_PATH):
        preds = {r["id"]: r for r in submission_records}
        refs = {json.loads(l)["id"]: json.loads(l) for l in open(ANSWERS_PATH, encoding="utf-8-sig") if l.strip() and json.loads(l)["id"] in preds}
        metrics = compute_pa(preds, refs)
        total = metrics["total"]
        correct = metrics["correct"]
        wrong = metrics["wrong"]
        abstain = metrics["abstain"]
        pa = metrics["pa"]
        
        print("\n" + "=" * 60)
        print("🏆 DATAMATE OFFICIAL BENCHMARK SCORE (EVALUATED SUBSET)")
        print("=" * 60)
        print(f"Total Questions  : {total}")
        print(f"Correct (+1 pt)  : {correct} ({correct/total*100:.1f}%)")
        print(f"Wrong   (-1 pt)  : {wrong} ({wrong/total*100:.1f}%)")
        print(f"Abstained (0 pt) : {abstain} ({abstain/total*100:.1f}%)")
        print("-" * 60)
        print(f"🔥 Penalized Accuracy (PA): {pa:+.4f} ({pa*100:+.2f}%)")
        print("=" * 60)
        
        # Detailed breakdown by question type
        print("\n📊 DETAILED BREAKDOWN BY QUESTION TYPE:")
        type_rows = []
        for t, s in sorted(metrics.get("per_type", {}).items()):
            tot = s["total"]
            cor = s["correct"]
            wro = s["wrong"]
            abs_ = s["abstain"]
            sub_pa = (cor - wro) / tot if tot > 0 else 0.0
            type_rows.append({
                "Type": t,
                "Total": tot,
                "Correct": cor,
                "Wrong": wro,
                "Abstain": abs_,
                "PA": f"{sub_pa:+.4f}",
                "Acc%": f"{cor/tot*100:.1f}%" if tot > 0 else "0.0%"
            })
        df_types = pd.DataFrame(type_rows)
        print(df_types.to_string(index=False))
        print("=" * 60)
    else:
        print("\n" + "=" * 60)
        print("ℹ️ Mode: PUBLIC TEST (Ground truth answers are hidden on Zalo server)")
        print("=" * 60)


## 7. Diagnostic Inspector: Error Analysis Breakdown
Generates `error_analysis.csv` and `abstain_analysis.csv` enriched with business ID and question text.


In [ ]:
import os
import json
import pandas as pd
from src.evaluate import score_answer

if os.path.exists(ANSWERS_PATH):
    questions_map = {}
    if os.path.exists(QUESTIONS_FILE):
        with open(QUESTIONS_FILE, "r", encoding="utf-8-sig") as f:
            for line in f:
                if line.strip():
                    q = json.loads(line)
                    questions_map[q["id"]] = q
                    
    gt_map = {}
    with open(ANSWERS_PATH, "r", encoding="utf-8-sig") as f:
        for line in f:
            if line.strip():
                ans = json.loads(line)
                gt_map[ans["id"]] = ans

    sub_map = {}
    with open(SUBMISSION_PATH, "r", encoding="utf-8-sig") as f:
        for line in f:
            if line.strip():
                sub = json.loads(line)
                sub_map[sub["id"]] = sub

    errors = []
    abstains = []
    for q_id, gt in gt_map.items():
        if q_id not in sub_map:
            continue
        sub = sub_map[q_id]
        score = score_answer(sub, gt)
        q_meta = questions_map.get(q_id, {})
        
        row = {
            "id": q_id,
            "business_id": q_meta.get("business_id", ""),
            "question": q_meta.get("question", ""),
            "pred_type": sub.get("type"),
            "gt_type": gt.get("type"),
            "pred_val": sub.get("value"),
            "gt_val": gt.get("value"),
            "as_of_date": q_meta.get("as_of_date", "")
        }
        if score < 0:
            errors.append(row)
        elif score == 0:
            abstains.append(row)
            
    df_err = pd.DataFrame(errors)
    df_abs = pd.DataFrame(abstains)
    
    err_csv = os.path.join(OUTPUT_DIR, "error_analysis.csv")
    abs_csv = os.path.join(OUTPUT_DIR, "abstain_analysis.csv")
    df_err.to_csv(err_csv, index=False, encoding="utf-8-sig")
    df_abs.to_csv(abs_csv, index=False, encoding="utf-8-sig")
    print(f"📊 Saved error analysis report   : {err_csv} ({len(df_err)} errors)")
    print(f"📊 Saved abstain analysis report : {abs_csv} ({len(df_abs)} abstains)")


## 8. Package & Archive Results
Compresses all outputs, logs, and evaluation reports into a single zip archive for 1-click download.


In [ ]:
import shutil

archive_path = os.path.join(WORKDIR, "results_dual_5090")
shutil.make_archive(archive_path, "zip", OUTPUT_DIR)
print(f"📦 Successfully created archive: {archive_path}.zip")
